# dfs-three-set-toposort — faded example 1: Topological sort of a tree with branching factor 3

> Faded drill from [Delta Drills](https://delta-drills.vercel.app). Atom: `dfs-three-set-toposort`. Running the beacon reports progress on the `Backprop: DFS three-set toposort` subtopic.

**Most of the solution is filled in — complete the one blanked step**, run the test, then fire the beacon. Less scaffolding than the worked example, more than the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper.
    `.grad` accumulates the leaf gradient at the end of the reverse pass."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
        self.grad = None
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: DFS three-set toposort` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`dfs-three-set-toposort`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "dfs-three-set-toposort"
DD_SUBTOPIC = "Backprop: DFS three-set toposort"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Concept

In the three-set DFS toposort, the `perm` set tracks fully processed nodes and causes `visit` to return immediately on a second visit. Without this check, nodes reachable by multiple paths would be emitted more than once. The `temp` set flags nodes currently on the recursion stack — a second entry to a `temp` node signals a cycle.

## Faded exercise 1

Complete `topological_sort_tree` below. The graph is a simple tree: a root with three children, each child having two leaf children (10 nodes total). Fill in the inner `visit` function body — specifically the guard checks and the post-order append.

**Fill in:** The complete body of the `visit` inner function, including the perm/temp guards, the recursive child loop, and the post-order append.

In [ ]:
def topological_sort_tree(root, get_children):
    result = []
    perm = set()
    temp = set()

    def visit(node):
        raise NotImplementedError()  # TODO: The complete body of the `visit` inner function, including the perm/temp guards, the recursive child loop, and the post-order append.

    visit(root)
    return result

# Quick smoke-test (not the graded test)
class N:
    def __init__(self, name, kids=None): self.name = name; self.kids = kids or []
    def __repr__(self): return self.name

leaves = [N(f'L{i}') for i in range(6)]
mids = [N(f'M{i}', leaves[2*i:2*i+2]) for i in range(3)]
root = N('R', mids)
result = topological_sort_tree(root, lambda n: n.kids)
print(result)
print('Root last:', result[-1] is root)


def _test():
    class N:
        def __init__(self, name, kids=None): self.name = name; self.kids = kids or []
        def __repr__(self): return self.name

    leaves = [N(f'L{i}') for i in range(6)]
    mids = [N(f'M{i}', leaves[2*i:2*i+2]) for i in range(3)]
    root = N('R', mids)

    result = topological_sort_tree(root, lambda n: n.kids)

    # 10 nodes (1 root + 3 mids + 6 leaves), each exactly once
    assert len(result) == 10
    assert len(set(id(n) for n in result)) == 10

    # root is last
    assert result[-1] is root

    # post-order: every node appears after all of its own children
    for i, m in enumerate(mids):
        for leaf in leaves[2*i:2*i+2]:
            assert result.index(leaf) < result.index(m)

    # every mid appears before root
    mid_positions = [result.index(m) for m in mids]
    assert max(mid_positions) < result.index(root)


try:
    _test()
    _dd_passed.add('faded1')
    print('[Delta Drills] faded1 passed.')
except AssertionError as _e:
    print('Test failed:', _e)

## Report completion

Run the cell below to report progress. The beacon fires only if the test above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'faded1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:faded1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()

<details><summary>Solution</summary>

```python
def topological_sort_tree(root, get_children):
    result = []
    perm = set()
    temp = set()

    def visit(node):
        nid = id(node)
        if nid in perm:
            return
        if nid in temp:
            raise ValueError(f'Cycle at {node!r}')
        temp.add(nid)
        for child in get_children(node):
            visit(child)
        temp.discard(nid)
        perm.add(nid)
        result.append(node)

    visit(root)
    return result

# Quick smoke-test (not the graded test)
class N:
    def __init__(self, name, kids=None): self.name = name; self.kids = kids or []
    def __repr__(self): return self.name

leaves = [N(f'L{i}') for i in range(6)]
mids = [N(f'M{i}', leaves[2*i:2*i+2]) for i in range(3)]
root = N('R', mids)
result = topological_sort_tree(root, lambda n: n.kids)
print(result)
print('Root last:', result[-1] is root)
```
</details>